# 06 Building the rationale annotation sheets

CSE 4889 Machine Learning: Explainable Multi-Label Hate Speech Detection in Transliterated Bangla

To check whether the LIME and SHAP explanations make sense, we need words that people marked as the reason a comment is hateful (rationales). This notebook picks 100 hate comments from the test set and makes one Excel sheet per annotator. The annotators mark the hateful words in Google Sheets, working independently.

Two team members annotate (Annotator A and Annotator B). No AI suggestions are shown in the sheet, so all marks come only from the annotators.

Runtime: CPU is fine (under a minute).

## Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports, paths and seed

In [2]:
import os
import json
import random
from datetime import datetime

import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

BASE = '/content/drive/MyDrive/ML_Project'
SEED = 42
NOTEBOOK_NAME = '06_rationale_sheet.ipynb'
PROC_DIR = os.path.join(BASE, 'data', 'processed')
XPLAIN_DIR = os.path.join(BASE, 'data', 'banth_xplain')
TAB_DIR = os.path.join(BASE, 'results', 'tables')
ASSETS_PATH = os.path.join(BASE, 'results', 'ASSETS.md')
random.seed(SEED)
np.random.seed(SEED)

CATEGORIES = ['Political', 'Religious', 'Gender', 'Personal Offense',
              'Abusive/Violence', 'Origin', 'Body Shaming', 'Misc']
ANNOTATORS = ['Ifta Faisal', 'Diab Alam']
N_COMMENTS = 100
MAX_WORDS = 30

## Helper functions

Save tables as CSV and LaTeX, and add files to ASSETS.md.

In [3]:
def save_table(df, name, float_format='%.2f'):
    df.to_csv(os.path.join(TAB_DIR, f'{name}.csv'), index=False)
    df.to_latex(os.path.join(TAB_DIR, f'{name}.tex'), index=False, escape=True,
                float_format=float_format, na_rep='--')
    print('Saved:', name, '(.csv and .tex)')

def register_asset(asset_id, file_name, what, notebook, section, caption):
    row = f'| {asset_id} | {file_name} | {what} | {notebook} | {section} | {caption} |'
    with open(ASSETS_PATH, encoding='utf-8') as f:
        lines = f.read().splitlines()
    lines = [l for l in lines if not l.startswith(f'| {asset_id} |')]
    lines.append(row)
    with open(ASSETS_PATH, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')
    print('Registered', asset_id, '->', file_name)

## Pick 100 hate comments from the test set

We use the test set because the model never saw these comments during training. Only hate comments are used, since rationales explain why something is hateful. We keep comments with 3 to 30 words so they fit in one row of the sheet.

If we picked at random, almost all comments would be Personal Offense. So we set a quota for each category, starting with the rarest, and fill any leftover places at random. The order is then shuffled (seed 42), so the categories are mixed in the sheet.

In [4]:
test_df = pd.read_csv(os.path.join(PROC_DIR, 'test.csv'), keep_default_na=False)
pool = test_df[(test_df['Label'] == 1) & test_df['word_count'].between(3, MAX_WORDS)]

QUOTA = {'Body Shaming': 8, 'Religious': 10, 'Origin': 10, 'Gender': 10, 'Misc': 10,
         'Political': 15, 'Abusive/Violence': 15, 'Personal Offense': 22}
chosen = []
for cat, q in QUOTA.items():
    avail = pool[(pool[cat] == 1) & ~pool['comment_id'].isin(chosen)]
    chosen += avail.sample(min(q, len(avail)), random_state=SEED)['comment_id'].tolist()
if len(chosen) < N_COMMENTS:
    rest = pool[~pool['comment_id'].isin(chosen)]
    chosen += rest.sample(N_COMMENTS - len(chosen), random_state=SEED)['comment_id'].tolist()
chosen = chosen[:N_COMMENTS]

sample = (pool.set_index('comment_id').loc[chosen].reset_index()
              .sample(frac=1, random_state=SEED).reset_index(drop=True))
sample['words'] = sample['clean_text'].str.split()
sample['categories'] = sample[CATEGORIES].apply(
    lambda r: ', '.join(c for c in CATEGORIES if r[c] == 1), axis=1)
print(len(sample), 'comments picked')
sample[['comment_id', 'categories', 'clean_text']].head(10)

100 comments picked


,comment_id,categories,clean_text
0,test_02013,"Personal Offense, Abusive/Violence",Police Maderchud khankirput mor tora magibaj 😡😡😡😡
1,test_03324,"Political, Abusive/Violence","Alhamdulillah, Kuttaliger sob gulo ke dhore di..."
2,test_01770,Abusive/Violence,Kuttar baccha ei shahosh koi paili? Ore neri k...
3,test_00179,Misc,Agey apnader mordhe indian gulare khuje ber koren
4,test_03028,Misc,Hate this lockdown vaccine nibo na
5,test_02052,Misc,Er confidence dekhe matha noshto
6,test_02938,"Political, Origin","Nah, tomra India banate chaichila..... amra 15..."
7,test_03557,Personal Offense,Amn vabe titel dey....shalar kisui bolar nai.....
8,test_00107,Religious,Hasina r fasi chai khoni hasina Muslim r satru ❤
9,test_01873,Body Shaming,Takar thakle koto cravings baire jai motki ekta


## How many comments of each category are in the sample (T17)

One comment can have more than one category, so the counts add up to more than 100. The table also shows how many hate comments of each category the whole test set has, so you can see how much the rare categories were oversampled.

In [5]:
hate_test = test_df[test_df['Label'] == 1]
composition = pd.DataFrame({
    'Category': CATEGORIES,
    'In rationale sample': [int(sample[c].sum()) for c in CATEGORIES],
    'All hate test comments': [int(hate_test[c].sum()) for c in CATEGORIES],
})
composition.loc[len(composition)] = ['Total comments', len(sample), len(hate_test)]
display(composition)
save_table(composition, 'tab_17_rationale_sample')
register_asset('T17', 'tables/tab_17_rationale_sample.csv (+ .tex)',
               'Category make-up of the 100-comment rationale sample vs all hate test comments',
               NOTEBOOK_NAME, 'III. Dataset (mini rationale set)',
               'TABLE X. Hate categories in the rationale-annotated sample compared with all hate comments in the test split. Rare categories were oversampled.')

,Category,In rationale sample,All hate test comments
0,Political,19,117
1,Religious,10,18
2,Gender,12,23
3,Personal Offense,45,715
4,Abusive/Violence,24,304
5,Origin,10,23
6,Body Shaming,8,14
7,Misc,10,30
8,Total comments,100,1062


Saved: tab_17_rationale_sample (.csv and .tex)
Registered T17 -> tables/tab_17_rationale_sample.csv (+ .tex)


## Annotation guidelines

These rules go on the first tab of each sheet, and are also saved as a text file for the report. They follow the idea of the HateXplain dataset: mark only the words that are the reason a comment is hateful.

In [6]:
GUIDELINES = [
    'Content warning: these are real hateful comments. Take breaks, and stop if it becomes upsetting.',
    '',
    'Task: every comment on the Annotation tab is hate speech. Mark the words that make it hateful.',
    '',
    '1. Read the whole comment first. The grey row shows the comment one word per cell.',
    '2. In the white row under it, type 1 under every word that is a reason the comment is hateful:',
    '   insults, slurs, threats, abusive words, or words attacking a person or group.',
    '3. Mark only the words that are needed. If a phrase is needed, mark every word of the phrase.',
    '4. Do not mark a name of a person, party or group just because it appears. Mark it only if it is used as an insult.',
    '5. Punctuation or emoji stuck to a word count as part of that word.',
    '6. If you think nothing in the comment is hateful, type 1 in the "Nothing hateful" column and leave the words empty.',
    '7. The Hate categories column shows the dataset labels. Use it as context only.',
    '8. Work alone. Do not look at or discuss the other person\'s sheet until both are finished.',
    '9. Do not change the grey rows, the IDs or the order of the rows.',
    '',
    'A marked cell turns red. Leave unmarked cells empty (no 0 needed).',
]
with open(os.path.join(XPLAIN_DIR, 'annotation_guidelines.txt'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(GUIDELINES) + '\n')
print('\n'.join(GUIDELINES))

Content warning: these are real hateful comments. Take breaks, and stop if it becomes upsetting.

Task: every comment on the Annotation tab is hate speech. Mark the words that make it hateful.

1. Read the whole comment first. The grey row shows the comment one word per cell.
2. In the white row under it, type 1 under every word that is a reason the comment is hateful:
   insults, slurs, threats, abusive words, or words attacking a person or group.
3. Mark only the words that are needed. If a phrase is needed, mark every word of the phrase.
4. Do not mark a name of a person, party or group just because it appears. Mark it only if it is used as an insult.
5. Punctuation or emoji stuck to a word count as part of that word.
6. If you think nothing in the comment is hateful, type 1 in the "Nothing hateful" column and leave the words empty.
7. The Hate categories column shows the dataset labels. Use it as context only.
8. Work alone. Do not look at or discuss the other person's sheet until 

## Build one Excel sheet per annotator

Each file has two tabs: Instructions and Annotation. On the Annotation tab every comment takes two rows:
- a grey row with the number, ID, categories and one word per cell,
- a white row under it where the annotator types 1 under the hateful words.

Only 1 can be typed in the white cells, and a cell with 1 turns red, so mistakes are easy to see. The first columns and the header row stay fixed when scrolling. Both files have exactly the same comments in the same order.

In [7]:
GREY = PatternFill('solid', fgColor='EDEDED')
RED = PatternFill('solid', fgColor='F4B6B6')
HEAD = PatternFill('solid', fgColor='D9E6F7')
LINE = Border(bottom=Side(style='thin', color='999999'))
FIRST_WORD_COL = 5
last_col = FIRST_WORD_COL + MAX_WORDS - 1

def build_workbook(annotator):
    wb = Workbook()
    ins = wb.active
    ins.title = 'Instructions'
    ins['A1'] = f'Annotator: {annotator}'
    ins['A1'].font = Font(bold=True, size=12)
    for r, line in enumerate(GUIDELINES, start=3):
        ins.cell(r, 1, line).alignment = Alignment(wrap_text=True)
    ins.column_dimensions['A'].width = 120

    ws = wb.create_sheet('Annotation')
    headers = ['No.', 'ID', 'Hate categories', 'Nothing hateful'] + [f'W{i}' for i in range(1, MAX_WORDS + 1)]
    for c, h in enumerate(headers, start=1):
        cell = ws.cell(1, c, h)
        cell.font, cell.fill = Font(bold=True), HEAD

    only_one = DataValidation(type='whole', operator='equal', formula1='1', allow_blank=True,
                              showErrorMessage=True, error='Type 1 to mark a word, or leave it empty.')
    ws.add_data_validation(only_one)

    for k, rec in sample.iterrows():
        wr, mr = 2 + 2 * k, 3 + 2 * k
        for c, v in enumerate([k + 1, rec['comment_id'], rec['categories'], ''], start=1):
            ws.cell(wr, c, v).fill = GREY
        for j, w in enumerate(rec['words']):
            ws.cell(wr, FIRST_WORD_COL + j, w).fill = GREY
        only_one.add(f'D{mr}')
        only_one.add(f'{get_column_letter(FIRST_WORD_COL)}{mr}:{get_column_letter(FIRST_WORD_COL + len(rec["words"]) - 1)}{mr}')
        for c in range(1, last_col + 1):
            ws.cell(mr, c).border = LINE

    last_row = 3 + 2 * (len(sample) - 1)
    ws.conditional_formatting.add(f'D2:{get_column_letter(last_col)}{last_row}',
                                  CellIsRule(operator='equal', formula=['1'], fill=RED))
    ws.column_dimensions['A'].width = 5
    ws.column_dimensions['B'].width = 12
    ws.column_dimensions['C'].width = 22
    ws.column_dimensions['D'].width = 10
    for c in range(FIRST_WORD_COL, last_col + 1):
        ws.column_dimensions[get_column_letter(c)].width = 13
    ws.freeze_panes = ws.cell(2, FIRST_WORD_COL)
    return wb

for name in ANNOTATORS:
    path = os.path.join(XPLAIN_DIR, f"rationale_sheet_{name.replace(' ', '_')}.xlsx")
    if os.path.exists(path):
        print('Already exists, NOT overwritten (it may contain annotations):', path)
        continue
    build_workbook(name).save(path)
    print('Saved:', path)

Saved: /content/drive/MyDrive/ML_Project/data/banth_xplain/rationale_sheet_Ifta_Faisal.xlsx
Saved: /content/drive/MyDrive/ML_Project/data/banth_xplain/rationale_sheet_Diab_Alam.xlsx


## Save the sample list

rationale_sample.csv keeps the ID, text, words and labels of the 100 comments in sheet order. The evaluation notebook uses it to match each mark in the sheets to the right word.

In [8]:
keep = ['comment_id', 'clean_text', 'categories', 'Label'] + CATEGORIES
out = sample[keep].copy()
out.insert(0, 'sheet_no', range(1, len(out) + 1))
out['words'] = sample['words'].map(lambda w: json.dumps(w, ensure_ascii=False))
out.to_csv(os.path.join(XPLAIN_DIR, 'rationale_sample.csv'), index=False, encoding='utf-8')
print('Saved rationale_sample.csv with', len(out), 'comments')
register_asset('M10', '../data/banth_xplain/rationale_sample.csv + annotation_guidelines.txt',
               'The 100 sampled test comments in sheet order, and the annotation guidelines',
               NOTEBOOK_NAME, 'III. Dataset (annotation protocol)',
               'Not a figure; source for the annotation protocol description.')

Saved rationale_sample.csv with 100 comments
Registered M10 -> ../data/banth_xplain/rationale_sample.csv + annotation_guidelines.txt


## What the annotators do now

1. In Google Drive, open ML_Project/data/banth_xplain/.
2. Open your own file (one rationale_sheet_<name>.xlsx per annotator). It opens in Google Sheets, and you can edit the Excel file directly. Don't use File > Save as Google Sheets, just edit the file as it is.
3. Read the Instructions tab, then mark words on the Annotation tab. Changes save automatically.
4. When both are finished, run the evaluation notebook.

Don't run this notebook again after annotation has started. It won't overwrite existing sheets, but a changed sample would no longer match them.

## Remember this

- What we did: picked 100 hate comments from the test set, with rare categories oversampled, and made one annotation sheet for each of the two annotators.
- Why: to judge whether LIME and SHAP explanations are sensible, we need words that people independently marked as the reason for the hate.
- Key number: 100 comments, 2 independent annotators, no AI suggestions in the sheets.